# Методология оценки и тестовый набор

## Формирование обучающей и тестовой выборки

Для обучения модели сравнения записей использовался подход на основе `party_public_id`. Записи с одинаковым идентификатором рассматривались как потенциальные дубликаты и использовались для формирования положительных пар.

Положительные пары формировались случайным образом внутри одного кластера. Отрицательные пары создавались двумя способами:
- случайные пары между различными сущностями;
- сложные отрицательные пары — похожие записи из разных кластеров, отобранные с помощью блокировки и текстового сходства.

Такой подход позволил получить как простые, так и сложные примеры для обучения модели.

Для уменьшения количества сравнений использовалась многоэтапная стратегия блокировки:
- префиксы имен;
- длина имени;
- отсортированные токены;
- страна;
- дополнительные текстовые ключи.

После формирования кандидатных пар для каждой пары вычислялись признаки:
- строковое сходство;
- пересечение токенов;
- совпадение страны;
- разница длин строк;
- дополнительные текстовые признаки.

На этих признаках обучалась модель `RandomForestClassifier`, которая оценивала вероятность того, что две записи являются дубликатами.

---

## Построение итоговой системы дедупликации

Полная система дедупликации состояла из нескольких этапов:

1. Предобработка данных:
   - нормализация текста;
   - транслитерация;
   - стандартизация юридических форм;

2. blocking:
   - формирование кандидатных пар для сравнения;

3. Сравнение записей:
   - вычисление вероятности дубликата моделью Random Forest;

4. Кластеризация:
   - построение графа связей между дубликатами;
   - объединение связанных записей в сущности;

5. Формирование итоговых сущностей:
   - назначение `cluster_id`;
   - создание канонических записей.

Полный алгоритм был применен к набору данных размером более 3.6 млн записей. В результате было получено около 1.71 млн уникальных сущностей.

---

## Размеченный тестовый набор

Для итоговой оценки качества был сформирован отдельный тестовый набор.

Из итогового дедуплицированного набора случайным образом отбирались:
- пары записей внутри одного кластера;
- похожие записи из разных кластеров.

После этого пары проходили ручную проверку. Разметка выполнялась по нормализованным и транслитерированным именам (`name_latin`). Для каждой пары вручную определялось:
- относятся ли записи к одной сущности (`label = 1`);
- либо являются разными сущностями (`label = 0`).

Неоднозначные случаи помечались отдельно и исключались из финальной оценки.

Итоговый вручную размеченный тестовый набор содержал 300 многоязычных пар.

---

## Обоснование выбранной методологии

Выбранный подход соответствует типичной схеме решения задач дедупликации и сопоставления сущностей.

Использование автоматически сформированных обучающих данных позволило обучить модель на большом количестве примеров без необходимости вручную размечать миллионы записей.

Одновременно была проведена ручная разметка отдельного тестового набора, что позволило получить независимую и более достоверную оценку качества всей системы.

Оценка выполнялась не только для модели сравнения пар, но и для всей итоговой системы дедупликации, включая:
- предобработку;
- блокировку;
- сравнение записей;
- кластеризацию.

Для оценки использовались метрики:
- Precision;
- Recall;
- F1-score;
- confusion matrix;
- анализ ошибок.

На вручную размеченном тестовом наборе система показала следующие результаты:
- Precision = 0.973;
- Recall = 0.896;
- F1-score = 0.933.